# Partie I — MLP sur Adult Income

## 01. Importation des bibliothèques

In [34]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

##  02. Reproductibilité et device CPU/GPU
Cette cellule prépare les bases de notre modèle en remplissant deux objectifs principaux :

1. **Assurer la reproductibilité (`set_seed`)** : En fixant la "graine" (seed) pour Python, NumPy et PyTorch, nous garantissons que les processus aléatoires (comme l'initialisation des poids ou le mélange des données) seront exactement les mêmes à chaque exécution. Cela rend les résultats constants et comparables.
2. **Assigner le matériel de calcul (`device`)** : Le code détecte automatiquement si une carte graphique (GPU / CUDA) est disponible pour accélérer massivement les calculs. Si aucune n'est trouvée, il utilisera le processeur classique (CPU) par défaut.

In [35]:
SEED = 42 
def set_seed(seed = 42): 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


## 03. Chargement du dataset Adult Income

In [36]:
adult = fetch_openml(name="adult", version=2, as_frame=True)

X = adult.data.copy()
y = adult.target.copy()

print("Dimensions X :", X.shape)
print("Dimensions y :", y.shape)
X.head()

Dimensions X : (48842, 14)
Dimensions y : (48842,)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States


## 04. Compréhension et analyse du dataset

In [37]:
print("Colonnes du dataset :")
print(X.columns.tolist())

print("\nTypes des variables :")
print(X.dtypes)

Colonnes du dataset :
['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country']

Types des variables :
age                  int64
workclass         category
fnlwgt               int64
education         category
education-num        int64
marital-status    category
occupation        category
relationship      category
race              category
sex               category
capital-gain         int64
capital-loss         int64
hours-per-week       int64
native-country    category
dtype: object


In [38]:
y.dtypes

CategoricalDtype(categories=['<=50K', '>50K'], ordered=False, categories_dtype=str)

In [40]:
display(X.info())

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   age             48842 non-null  int64   
 1   workclass       46043 non-null  category
 2   fnlwgt          48842 non-null  int64   
 3   education       48842 non-null  category
 4   education-num   48842 non-null  int64   
 5   marital-status  48842 non-null  category
 6   occupation      46033 non-null  category
 7   relationship    48842 non-null  category
 8   race            48842 non-null  category
 9   sex             48842 non-null  category
 10  capital-gain    48842 non-null  int64   
 11  capital-loss    48842 non-null  int64   
 12  hours-per-week  48842 non-null  int64   
 13  native-country  47985 non-null  category
dtypes: category(8), int64(6)
memory usage: 2.6 MB


None

In [44]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Variables numériques :", numeric_features)
print("Nombre :", len(numeric_features))

print("\nVariables catégorielles :", categorical_features)
print("Nombre :", len(categorical_features))

Variables numériques : ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
Nombre : 6

Variables catégorielles : ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']
Nombre : 8


In [43]:
display(y.value_counts(normalize=True) * 100)

class
<=50K    76.071823
>50K     23.928177
Name: proportion, dtype: float64

In [39]:
print("\nDistribution de la variable cible :")
df = y.value_counts().rename_axis('class').reset_index(name='count')
df['percentage %'] = (df['count'] / df['count'].sum() * 100).round(2)
display(df)


Distribution de la variable cible :


,class,count,percentage %
0,<=50K,37155,76.07
1,>50K,11687,23.93


In [18]:
print("Valeurs manquantes classiques :")
print(X.isna().sum())

Valeurs manquantes classiques :
age                  0
workclass         2799
fnlwgt               0
education            0
education-num        0
marital-status       0
occupation        2809
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     857
dtype: int64


In [52]:
print(f"\nValeurs dupliquées : {X.duplicated().sum()}")
print(f"\nValeurs manquantes (y compris les '?') : {X.isnull().sum().sum()}")


Valeurs dupliquées : 57

Valeurs manquantes (y compris les '?') : 6465


In [53]:
display(X.describe(include='all'))

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
count,48842.000000,46043,4.884200e+04,48842,48842.000000,48842,46033,48842,48842,48842,48842.000000,48842.000000,48842.000000,47985
unique,NaN,8,NaN,16,NaN,7,14,6,5,2,NaN,NaN,NaN,41
top,NaN,Private,NaN,HS-grad,NaN,Married-civ-spouse,Prof-specialty,Husband,White,Male,NaN,NaN,NaN,United-States
freq,NaN,33906,NaN,15784,NaN,22379,6172,19716,41762,32650,NaN,NaN,NaN,43832
mean,38.643585,NaN,1.896641e+05,NaN,10.078089,NaN,NaN,NaN,NaN,NaN,1079.067626,87.502314,40.422382,NaN
std,13.710510,NaN,1.056040e+05,NaN,2.570973,NaN,NaN,NaN,NaN,NaN,7452.019058,403.004552,12.391444,NaN
min,17.000000,NaN,1.228500e+04,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,1.000000,NaN
25%,28.000000,NaN,1.175505e+05,NaN,9.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,40.000000,NaN
50%,37.000000,NaN,1.781445e+05,NaN,10.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,40.000000,NaN
75%,48.000000,NaN,2.376420e+05,NaN,12.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,45.000000,NaN


### Distribution des classes
On regarde `y` qui contient la cible et permet de verifier l'equilibre des classes.
- ` remarque : On evite  ``X.value_counts()`` sur la matrice de features: sur un tableau tres large, cela peut exploser la memoire. `

In [54]:
# Ne pas appliquer value_counts sur X : sur une matrice tres large, cela peut provoquer un MemoryError.
display(y.value_counts(normalize=True) * 100)

class
<=50K    76.071823
>50K     23.928177
Name: proportion, dtype: float64

## 05. Nettoyage des données

In [56]:
print("Valeurs manquantes classiques :")
print(X.isna().sum())

print("\nRecherche des '?' dans les colonnes catégorielles :")
for col in categorical_features:
    count_question = (X[col].astype(str).str.strip() == "?").sum()
    print(col, ":", count_question)

Valeurs manquantes classiques :
age                  0
workclass         2799
fnlwgt               0
education            0
education-num        0
marital-status       0
occupation        2809
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     857
dtype: int64

Recherche des '?' dans les colonnes catégorielles :
workclass : 0
education : 0
marital-status : 0
occupation : 0
relationship : 0
race : 0
sex : 0
native-country : 0


In [57]:
df = X.copy()
df["income"] = y

# Supprimer les lignes contenant des valeurs manquantes
df_clean = df.dropna().copy()

# Nettoyer les espaces seulement après suppression des NaN
for col in df_clean.select_dtypes(include=["object", "category"]).columns:
    df_clean[col] = df_clean[col].astype(str).str.strip()

print("Dimensions avant nettoyage :", df.shape)
print("Dimensions après nettoyage :", df_clean.shape)

print("\nValeurs manquantes après nettoyage :")
print(df_clean.isna().sum())


Dimensions avant nettoyage : (48842, 15)
Dimensions après nettoyage : (45222, 15)

Valeurs manquantes après nettoyage :
age               0
workclass         0
fnlwgt            0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
income            0
dtype: int64


### Encodage de la cible :

In [67]:
print("\nValeurs uniques dans la variable cible :")
print(df_clean["income"].unique().tolist())



Valeurs uniques dans la variable cible :
['<=50K', '>50K']


In [68]:
print("\nDistribution de la variable cible après nettoyage :")
display(pd.DataFrame(df_clean["income"].value_counts()))

df_clean["target"] = df_clean["income"].map({
    "<=50K": 0,
    ">50K": 1
})
print("\nDistribution de la variable cible après encodage :")
display(pd.DataFrame(df_clean["target"].value_counts()))


Distribution de la variable cible après nettoyage :


,count
income,
<=50K,34014
>50K,11208



Distribution de la variable cible après encodage :


,count
target,
0,34014
1,11208


### Séparation features / target :

In [70]:
print(df_clean.columns.tolist())
df_clean.head()

['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income', 'target']


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income,target
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K,0
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K,0
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K,1
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K,1
5,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K,0


In [73]:
X_clean = df_clean.drop(columns=["income", "target"])
y_clean = df_clean["target"].astype(int)

numeric_features = X_clean.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_clean.select_dtypes(include=["object", "category","string"]).columns.tolist()

display("Variables numériques :", numeric_features)
display("Variables catégorielles :", categorical_features)

'Variables numériques :'

['age',
 'fnlwgt',
 'education-num',
 'capital-gain',
 'capital-loss',
 'hours-per-week']

'Variables catégorielles :'

['workclass',
 'education',
 'marital-status',
 'occupation',
 'relationship',
 'race',
 'sex',
 'native-country']

## 06. Split train / validation / test

`Important : on fait le split avant d’entraîner le préprocesseur pour éviter la fuite de données.`

In [76]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X_clean,
    y_clean,
    test_size=0.30,
    random_state=SEED,
    stratify=y_clean
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

print("Train :", X_train.shape, y_train.shape)
print("Validation :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)

Train : (31655, 14) (31655,)
Validation : (6783, 14) (6783,)
Test : (6784, 14) (6784,)


In [84]:
print("Distribution train ( % ) :")
display(y_train.value_counts(normalize=True) * 100 )

print("\nDistribution validation ( % ) :")
display(y_val.value_counts(normalize=True) * 100 )

print("\nDistribution test ( % ) :")
display(y_test.value_counts(normalize=True) * 100 )

Distribution train ( % ) :


target
0    75.214026
1    24.785974
Name: proportion, dtype: float64


Distribution validation ( % ) :


target
0    75.217455
1    24.782545
Name: proportion, dtype: float64


Distribution test ( % ) :


target
0    75.221108
1    24.778892
Name: proportion, dtype: float64

## 07. Prétraitement : encodage et normalisation

version avec normalisation :

On va standardiser les variables numériques, encoder les variables textuelles et applique ces changements sur nos ensembles d'entraînement, de validation et de test.

In [85]:
try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    print("Warning: 'sparse_output' parameter not available. Using 'sparse=False' instead.")
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor_scaled = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", onehot, categorical_features)
    ]
)

X_train_scaled = preprocessor_scaled.fit_transform(X_train)
X_val_scaled = preprocessor_scaled.transform(X_val)
X_test_scaled = preprocessor_scaled.transform(X_test)

print("X_train après prétraitement :", X_train_scaled.shape)

X_train après prétraitement : (31655, 104)


Version sans normalisation, pour l’expérience comparative :

In [86]:
try:
    onehot_no_scale = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    print("Warning: 'sparse_output' parameter not available. Using 'sparse=False' instead.")
    onehot_no_scale = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor_no_scale = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", onehot_no_scale, categorical_features)
    ]
)

X_train_no_scale = preprocessor_no_scale.fit_transform(X_train)
X_val_no_scale = preprocessor_no_scale.transform(X_val)
X_test_no_scale = preprocessor_no_scale.transform(X_test)

print("X_train sans normalisation :", X_train_no_scale.shape)

X_train sans normalisation : (31655, 104)


## 08. Création des Dataset et DataLoader PyTorch

### 🧠 Création du Dataset PyTorch personnalisé

Cette cellule définit une classe personnalisée appelée `AdultIncomeDataset`. Elle hérite de la classe `Dataset` de **PyTorch**. 
Son but est de convertir nos tableaux de données (NumPy/Pandas) en objets compatibles avec les outils d'entraînement de PyTorch (comme les `DataLoader`).

La classe surcharge les **trois méthodes indispensables** imposées par PyTorch :

1. **`__init__(self, X, y)` (Initialisation) :**
   * Reçoit les fonctionnalités de prédiction (`X`) et la variable cible (`y`).
   * Convertit les données en **Tenseurs PyTorch** de type décimal (`torch.float32`).
   * Utilise `.view(-1, 1)` sur `y` pour s'assurer que la variable cible a la forme d'une colonne verticale (vecteur 2D), ce qui est obligatoire pour calculer les fonctions de perte (*loss*) en classification binaire.

2. **`__len__(self)` (Taille) :**
   * Renvoie le nombre total d'exemples (lignes) présents dans le jeu de données. 
   * Permet à PyTorch de savoir quand une époque d'entraînement est terminée.

3. **`__getitem__(self, idx)` (Extraction) :**
   * Permet de récupérer un exemple précis (`X[idx]`) associé à son étiquette (`y[idx]`) via son index.
   * Cette méthode est appelée automatiquement en arrière-plan pour charger les données par petits groupes (mini-batches).


In [87]:
class AdultIncomeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.to_numpy(), dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

### 📦 Préparation des chargeurs de données (DataLoader) et taille d'entrée

Cette cellule prépare l'alimentation du modèle PyTorch en découpant les données et en configurant la structure du futur réseau de neurones.

#### 1. Configuration des Mini-Batches
* **`BATCH_SIZE = 128`** : Définit la taille des lots. Le modèle ne va pas analyser tout le jeu de données d'un coup, mais par groupes de 128 lignes. Cela stabilise l'entraînement et évite de saturer la mémoire (RAM).

#### 2. Instanciation des Datasets
* Les données d'entraînement (`scaled`), de validation et de test sont converties en objets PyTorch en utilisant la classe personnalisée `AdultIncomeDataset` créée à l'étape précédente.

#### 3. Création des `DataLoader`
Le `DataLoader` gère la logique d'envoi des données au modèle durant l'apprentissage :
* **`train_loader` (`shuffle=True`)** : Mélange aléatoirement les données d'entraînement à chaque époque. C'est indispensable pour que le modèle n'apprenne pas l'ordre des lignes par cœur et généralise mieux.
* **`val_loader` & `test_loader` (`shuffle=False`)** : Ne mélangent pas les données. Pour l'évaluation, l'ordre n'a pas d'importance, désactiver le mélange permet de gagner du temps de calcul.

#### 4. Dimension d'entrée du réseau (`input_dim`)
* **`X_train_scaled.shape[1]`** : Récupère le nombre exact de colonnes finales après notre étape de prétraitement (StandardScaler + OneHotEncoder). 
* Cette variable `input_dim` est cruciale car elle dictera la taille de la **toute première couche** de votre réseau de neurones.


In [88]:
BATCH_SIZE = 128

train_dataset = AdultIncomeDataset(X_train_scaled, y_train)
val_dataset = AdultIncomeDataset(X_val_scaled, y_val)
test_dataset = AdultIncomeDataset(X_test_scaled, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

input_dim = X_train_scaled.shape[1]
print("Nombre de features après encodage :", input_dim)

Nombre de features après encodage : 104


DataLoaders sans normalisation :

In [89]:
train_dataset_no_scale = AdultIncomeDataset(X_train_no_scale, y_train)
val_dataset_no_scale = AdultIncomeDataset(X_val_no_scale, y_val)
test_dataset_no_scale = AdultIncomeDataset(X_test_no_scale, y_test)

train_loader_no_scale = DataLoader(train_dataset_no_scale, batch_size=BATCH_SIZE, shuffle=True)
val_loader_no_scale = DataLoader(val_dataset_no_scale, batch_size=BATCH_SIZE, shuffle=False)
test_loader_no_scale = DataLoader(test_dataset_no_scale, batch_size=BATCH_SIZE, shuffle=False)